# Day 40: Benchmark Ops:Byte Ratio in Practice

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Empirically measures arithmetic intensity and throughput for matrix multiplications across sizes, overlaying results on the analytical roofline model. Verifies that small batch sizes (decode) are deeply memory-bound while large batches (prefill) approach compute peak.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Benchmark Ops:Byte Ratio Across Matrix Sizes


In [ ]:
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def measure_matmul(m, k, n, dtype=torch.float16, warmup=5, iters=50):
    A = torch.randn(m, k, dtype=dtype, device=device)
    B = torch.randn(k, n, dtype=dtype, device=device)

    for _ in range(warmup):
        C = A @ B
    if device == 'cuda': torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        C = A @ B
    if device == 'cuda': torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) / iters * 1000

    flops = 2 * m * k * n
    bytes_moved = (m*k + k*n + m*n) * 2  # FP16
    intensity = flops / bytes_moved
    tflops = flops / (t_ms / 1000) / 1e12
    return t_ms, intensity, tflops

print(f'Matrix multiply benchmark on {device}:')
print(f'{"M":>6}{"K":>6}{"N":>6} {"Intensity":>12} {"Time (ms)":>12} {"TF/s":>10}')
print('-' * 55)
cases = [
    (1, 4096, 14336),    # decode batch=1 (memory-bound)
    (8, 4096, 14336),    # decode batch=8
    (32, 4096, 14336),   # decode batch=32
    (128, 4096, 14336),  # decode batch=128
    (4096, 4096, 4096),  # prefill seq=4096 (compute-bound)
    (512, 512, 512),     # small matmul
]
for m, k, n in cases:
    t, intensity, tflops = measure_matmul(m, k, n)
    print(f'{m:>6}{k:>6}{n:>6} {intensity:>12.1f} {t:>12.3f} {tflops:>10.2f}')


## 2. Roofline Plot with Empirical Data


In [ ]:
# Collect empirical data and overlay on roofline model
intensities = []
tflops_vals = []
for m, k, n in cases:
    _, intensity, tflops = measure_matmul(m, k, n)
    intensities.append(intensity)
    tflops_vals.append(tflops)

# GPU specs (DGX Spark GB10 approximate)
peak_tflops = 67
hbm_bw = 0.273  # TB/s
ridge = peak_tflops / hbm_bw

xs = np.logspace(-1, 4, 300)
roofline = np.minimum(peak_tflops, xs * hbm_bw)

plt.figure(figsize=(10, 6))
plt.loglog(xs, roofline, 'b-', linewidth=2, label='Roofline')
plt.scatter(intensities, tflops_vals, s=100, color='red', zorder=5)
for (m,k,n), intensity, tflops in zip(cases, intensities, tflops_vals):
    plt.annotate(f'bs={m}', (intensity, tflops), xytext=(5,5), textcoords='offset points', fontsize=8)
plt.axvline(ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge point ({ridge:.0f} FLOP/B)')
plt.xlabel('Arithmetic Intensity (FLOP/B)')
plt.ylabel('Attainable TFLOP/s')
plt.title('Roofline: Measured Matmul Performance')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('roofline_empirical.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments

1. Measure ops:byte ratio for attention kernels. Does causal masking affect the ratio?
2. Try FP32 vs FP16 vs BF16. How does dtype affect the measured throughput vs theoretical peak?
3. Compare measured ridge point against the spec sheet. How close does PyTorch get to peak HBM bandwidth?


## Key Takeaways

- See concept overview above for the key points.
- Day 40 complete.
